# Lab 2: RAG Pipeline Evolution -- From Keywords to Intelligence

**AI-2: AI Backend Engineering** | Northbrook Partners Document QA

---

**Weight:** 20% of course grade  
**Due:** Start of Session 4.2  
**Estimated time:** 2-3 hours

## The Evolution

In Session 3.2, you used keyword matching to route retrieval queries. It worked -- but it was
fragile. A question that doesn't contain the right keyword gets no filtering at all. In this lab,
you will replace keyword matching with an LLM classifier, add confidence-based routing, and
evaluate whether the added intelligence is worth the cost.

```
NAIVE RAG:        Question --> Embed --> Search ALL chunks ----------> Generate
KEYWORD FILTER:   Question --> Keyword match --> Filter chunks --> Search --> Generate
YOUR PIPELINE:    Question --> LLM Classify --> Confidence route --> Smart retrieve --> Generate
```

## Rubric

| Criterion | Points |
|-----------|--------|
| LLM Classifier Implementation | 25 |
| Pipeline Routing | 25 |
| Comparison Results | 20 |
| Written Analysis (Q1-Q6) | 30 |
| **Passing Grade** | **70% or above** |

## Section 0: Setup

In [ ]:
import sys
from pathlib import Path
from dotenv import load_dotenv

_here = Path(".").resolve()
_root = _here
while _root != _root.parent:
    if (_root / "pyproject.toml").exists():
        break
    _root = _root.parent
sys.path.insert(0, str(_root))
load_dotenv()

import json
import re
import anthropic
from pydantic import BaseModel, Field
from typing import Literal

from src.s4_retrieval.naive import naive_retrieve
from src.s4_retrieval.filtered import filtered_retrieve
from src.s0_generation.generate import call_claude_with_usage
from src.s2_embeddings.embed import embed_texts
from src.s3_ingestion.store import get_collection


def parse_json_response(text: str) -> dict:
    """Parse JSON from an LLM response, stripping markdown code fences if present."""
    cleaned = re.sub(r"```(?:json)?\s*", "", text).strip().rstrip("`")
    return json.loads(cleaned)


print("Setup complete.")

In [ ]:
collection = get_collection()
sample = collection.peek(limit=3)
print(f"Collection has {collection.count()} chunks")
print(f"\nMetadata fields: {list(sample['metadatas'][0].keys())}")
print(f"\nDocument types in corpus:")
all_meta = collection.get(include=["metadatas"])["metadatas"]
doc_types = {}
for m in all_meta:
    dt = m.get("doc_type", "unknown")
    doc_types[dt] = doc_types.get(dt, 0) + 1
for dt, count in sorted(doc_types.items()):
    print(f"  {dt}: {count} chunks")

## Section 1: The Keyword Baseline (PROVIDED)

In Session 3.2, you built `build_filter()` using keyword matching. Here it is again -- your baseline.
This is the approach you are trying to beat.

In [ ]:
def build_filter(question: str) -> dict | None:
    """Keyword-based filter from Session 3.2."""
    question_lower = question.lower()

    financial_keywords = ["revenue", "profit", "earnings", "financial", "q3", "q4", "quarterly"]
    if any(kw in question_lower for kw in financial_keywords):
        return {"doc_type": "financial"}

    meeting_keywords = ["board meeting", "standup", "committee", "discussed", "decided", "engineering sync"]
    if any(kw in question_lower for kw in meeting_keywords):
        return {"doc_type": "meeting"}

    memo_keywords = ["ceo", "announcement", "relocation", "security update", "kickoff", "memo"]
    if any(kw in question_lower for kw in memo_keywords):
        return {"doc_type": "memo"}

    policy_keywords = ["policy", "handbook", "vacation", "expense", "remote work", "dress code", "pto", "reimbursement"]
    if any(kw in question_lower for kw in policy_keywords):
        return {"doc_type": "policy"}

    return None

In [ ]:
eval_queries = [
    {"question": "What is Northbrook's vacation policy?", "expected_type": "policy"},
    {"question": "What were the Q3 revenue numbers?", "expected_type": "financial"},
    {"question": "What decisions were made at the board meeting?", "expected_type": "meeting"},
    {"question": "What did the CEO say about company direction?", "expected_type": "memo"},
    {"question": "What services does Northbrook offer?", "expected_type": None},
    {"question": "What is the expense reimbursement limit for hotels?", "expected_type": "policy"},
    {"question": "How is the cloud migration progressing?", "expected_type": None},
    {"question": "When is the office relocation happening?", "expected_type": "memo"},
]

print(f"Total queries: {len(eval_queries)}")
for i, q in enumerate(eval_queries):
    print(f"  {i+1}. [{q['expected_type'] or 'any'}] {q['question']}")

In [ ]:
print(f"{'#':<3} {'Question':<55} {'Expected':<12} {'Keyword':<12} {'Match'}")
print("-" * 90)

keyword_results = []
for i, q in enumerate(eval_queries):
    kw_filter = build_filter(q["question"])
    kw_type = kw_filter["doc_type"] if kw_filter else None
    match = kw_type == q["expected_type"]
    keyword_results.append({"question": q["question"], "expected": q["expected_type"], "keyword": kw_type, "match": match})
    print(f"{i+1:<3} {q['question'][:53]:<55} {str(q['expected_type']):<12} {str(kw_type):<12} {'Y' if match else 'N'}")

kw_accuracy = sum(1 for r in keyword_results if r["match"]) / len(keyword_results)
print(f"\nKeyword accuracy: {kw_accuracy:.0%}")

## Section 2: Build the LLM Classifier (25 pts)

Keyword matching is fragile. A question like *"How is the cloud migration progressing?"* has no
keyword match at all, even though a human could reason about where that information lives. An
LLM can understand intent, not just substring matches.

Your job: build an LLM-powered classifier that returns structured output via Pydantic.

In [ ]:
class QueryClassification(BaseModel):
    doc_type: Literal["financial", "meeting", "memo", "policy"] | None = Field(
        description="The document type most likely to contain the answer, or None if unclear"
    )
    reasoning: str = Field(
        description="Brief explanation of why this document type was chosen"
    )

In [ ]:
CLASSIFIER_PROMPT = """You are a query classifier for a document retrieval system.

Given a user's question, determine which document type is most likely to contain the answer.

Available document types:
- "financial": Quarterly reports, revenue data, budgets, earnings
- "meeting": Board meetings, standups, committee notes, decisions
- "memo": CEO announcements, office updates, security notices, company-wide communications
- "policy": HR policies, handbooks, vacation/expense/remote work rules

If the question does not clearly map to one type, return null for doc_type.

Respond with valid JSON matching this schema:
{"doc_type": "<type or null>", "reasoning": "<your explanation>"}"""

### Your Task: Implement `llm_classify()`

Use `call_claude_with_usage()` to send the question to Claude with the classifier prompt,
then parse the JSON response into a `QueryClassification` object.

**Hints:**
- `call_claude_with_usage(prompt, system_prompt, temperature)` returns a dict with a `"text"` key
- Use `parse_json_response()` (defined in setup) to parse the text -- it handles markdown code fences
- Handle the case where JSON contains `"null"` (string) vs `null` (JSON null)

In [ ]:
def llm_classify(question: str) -> QueryClassification:
    """Classify a question using Claude and return structured output.

    Args:
        question: The user's question

    Returns:
        QueryClassification with doc_type and reasoning
    """
    # Step 1: Call Claude with the classifier prompt and the question
    # Use call_claude_with_usage() with CLASSIFIER_PROMPT as system_prompt
    # and the question as the prompt. Temperature 0.0 for consistency.

    # YOUR CODE HERE


    # Step 2: Parse the JSON response into a QueryClassification object
    # The response text should be valid JSON. Parse it with json.loads()
    # then pass to QueryClassification(**parsed)
    # Handle the case where doc_type is "null" or null in JSON -> None in Python

    # YOUR CODE HERE


    return classification

In [ ]:
print(f"{'#':<3} {'Question':<55} {'LLM Type':<12} {'Reasoning'}")
print("-" * 110)

llm_results = []
for i, q in enumerate(eval_queries):
    result = llm_classify(q["question"])
    llm_results.append({
        "question": q["question"],
        "expected": q["expected_type"],
        "llm_type": result.doc_type,
        "reasoning": result.reasoning,
        "match": result.doc_type == q["expected_type"],
    })
    print(f"{i+1:<3} {q['question'][:53]:<55} {str(result.doc_type):<12} {result.reasoning[:60]}")

llm_accuracy = sum(1 for r in llm_results if r["match"]) / len(llm_results)
print(f"\nLLM accuracy: {llm_accuracy:.0%}")

In [ ]:
print(f"{'#':<3} {'Question':<45} {'Expected':<12} {'Keyword':<12} {'LLM':<12} {'Agree'}")
print("-" * 95)

for i in range(len(eval_queries)):
    kw = keyword_results[i]["keyword"]
    llm = llm_results[i]["llm_type"]
    exp = eval_queries[i]["expected_type"]
    agree = kw == llm
    print(
        f"{i+1:<3} {eval_queries[i]['question'][:43]:<45} "
        f"{str(exp):<12} {str(kw):<12} {str(llm):<12} {'Y' if agree else '-- DISAGREE'}"
    )

### Q1 (5 pts)

Compare the keyword classifier and the LLM classifier across all 8 queries. For cases where
they disagreed, which made the better call? What failure modes does each approach have that the
other doesn't?

*Your answer here*

## Section 3: Add Confidence Routing (25 pts)

Your classifier sometimes picks wrong. Before you code anything, think about this:

> If the model says "financial" with low confidence, what are your options? What is the cost of
> filtering too aggressively (you miss the answer) vs not filtering at all (you get noise)?
> Which failure is worse for a user?

You need to do three things:

1. **Update the schema** to include a confidence field
2. **Update the system prompt** to tell the model what confidence MEANS
3. **Build a routing function** that uses confidence to decide the retrieval strategy

### Step 1: The Updated Schema

This schema adds a `confidence` field. It is provided for you.

In [ ]:
class SmartClassification(BaseModel):
    doc_type: Literal["financial", "meeting", "memo", "policy"] | None = Field(
        description="The document type most likely to contain the answer, or None if unclear"
    )
    confidence: Literal["high", "medium", "low"] = Field(
        description="How confident the classification is"
    )
    reasoning: str = Field(
        description="Brief explanation of the classification and confidence level"
    )

### Step 2: Update the System Prompt

The model needs to know what high, medium, and low confidence mean. The original classifier
prompt is below -- modify it to include guidance for the confidence field.

In [ ]:
# ======================================================================
#  MODIFY THIS PROMPT
#  Add guidance for the confidence field.
#  The model needs to know what high/medium/low means.
# ======================================================================

SMART_CLASSIFIER_PROMPT = """You are a query classifier for a document retrieval system.

Given a user's question, determine which document type is most likely to contain the answer.

Available document types:
- "financial": Quarterly reports, revenue data, budgets, earnings
- "meeting": Board meetings, standups, committee notes, decisions
- "memo": CEO announcements, office updates, security notices, company-wide communications
- "policy": HR policies, handbooks, vacation/expense/remote work rules

If the question does not clearly map to one type, return null for doc_type.

# YOUR ADDITIONS HERE: Define what high, medium, and low confidence mean.
# Think about: What makes a classification obvious vs ambiguous?
# When should the model say "low"?

Respond with valid JSON matching this schema:
{"doc_type": "<type or null>", "confidence": "<high|medium|low>", "reasoning": "<your explanation>"}"""

### Step 3: Build `smart_retrieve()`

This is your architectural decision. Use the classification and confidence to decide how
to retrieve. Think about:

- What do you do when confidence is "high"?
- What do you do when confidence is "medium"?
- What do you do when confidence is "low"?
- What if doc_type is None?

There is no single right answer. Design your routing logic and justify it in Q2.

In [ ]:
def smart_retrieve(question: str, top_k: int = 5) -> dict:
    """Classify a question with confidence, then route to the right retrieval strategy.

    This is YOUR architecture decision. Use the classification and confidence
    to decide how to retrieve. Return everything needed to evaluate the decision.

    Args:
        question: The user's question
        top_k: Number of chunks to retrieve

    Returns:
        dict with keys: "question", "classification", "method",
        "sources", "reasoning"
    """
    # Step 1: Classify the question using your updated SmartClassification
    # (Similar to llm_classify but using SMART_CLASSIFIER_PROMPT and SmartClassification)

    # YOUR CODE HERE


    # Step 2: Route based on confidence
    # This is your architectural decision:
    # - What do you do when confidence is "high"?
    # - What do you do when confidence is "medium"?
    # - What do you do when confidence is "low"?
    # - What if doc_type is None?
    # Design your routing logic and implement it.

    # YOUR CODE HERE


    # Step 3: Build the result dict with full reasoning trail
    # Include: the classification object, what method was chosen and why,
    # the retrieved sources, and a human-readable reasoning string

    # YOUR CODE HERE


    return {
        "question": question,
        "classification": ...,   # the SmartClassification object (or dict)
        "method": ...,           # "filtered", "naive", "filtered_fallback", etc.
        "sources": ...,          # list of retrieved chunks
        "reasoning": ...,        # string explaining the full decision chain
    }

In [ ]:
print(f"{'#':<3} {'Question':<40} {'Type':<12} {'Conf':<8} {'Method':<20} {'Sources':<8} {'Reasoning'}")
print("-" * 130)

smart_results = []
for i, q in enumerate(eval_queries):
    result = smart_retrieve(q["question"], top_k=5)
    smart_results.append(result)

    cls = result["classification"]
    doc_type = cls.doc_type if hasattr(cls, "doc_type") else cls.get("doc_type")
    confidence = cls.confidence if hasattr(cls, "confidence") else cls.get("confidence")

    print(
        f"{i+1:<3} {q['question'][:38]:<40} {str(doc_type):<12} {str(confidence):<8} "
        f"{result['method']:<20} {len(result['sources']):<8} {result['reasoning'][:40]}"
    )

### Q2 (10 pts)

Explain your routing logic. What does your pipeline do for high, medium, and low confidence?
Why did you make those choices? Show a specific query where confidence routing made a different
decision than always-filtering. Was it the right call?

*Your answer here*

## Section 4: The Full Comparison (20 pts)

You now have three approaches. Let's run them head to head.

- **Naive:** No filtering at all -- pure semantic similarity
- **Keyword:** The `build_filter()` approach from Session 3.2
- **Smart:** Your LLM classifier with confidence-based routing

The comparison runner below is provided. It computes source precision for each approach:
what fraction of retrieved chunks have the expected `doc_type`.

In [ ]:
# Run all three approaches on every query
comparison_results = []

for q in eval_queries:
    question = q["question"]
    expected = q["expected_type"]

    # Approach 1: Naive (no filtering)
    naive_sources = naive_retrieve(question, top_k=5)
    naive_types = [s["metadata"].get("doc_type", "?") for s in naive_sources]
    naive_precision = (
        sum(1 for t in naive_types if t == expected) / len(naive_types)
        if naive_types and expected
        else None
    )

    # Approach 2: Keyword filtering
    kw_filter = build_filter(question)
    if kw_filter:
        kw_sources = filtered_retrieve(question, filters=kw_filter, top_k=5)
    else:
        kw_sources = naive_sources
    kw_types = [s["metadata"].get("doc_type", "?") for s in kw_sources]
    kw_precision = (
        sum(1 for t in kw_types if t == expected) / len(kw_types)
        if kw_types and expected
        else None
    )

    # Approach 3: Your smart pipeline
    smart_result = smart_retrieve(question, top_k=5)
    smart_sources = smart_result["sources"]
    smart_types = [s["metadata"].get("doc_type", "?") for s in smart_sources]
    smart_precision = (
        sum(1 for t in smart_types if t == expected) / len(smart_types)
        if smart_types and expected
        else None
    )

    comparison_results.append({
        "question": question,
        "expected": expected,
        "naive_precision": naive_precision,
        "kw_filter": kw_filter,
        "kw_precision": kw_precision,
        "smart_classification": smart_result["classification"],
        "smart_method": smart_result["method"],
        "smart_precision": smart_precision,
        "smart_reasoning": smart_result["reasoning"],
    })

In [ ]:
print(f"{'#':<3} {'Question':<40} {'Expected':<10} {'Naive':>7} {'Keyword':>8} {'Smart':>7}  {'Method'}")
print("-" * 110)

for i, r in enumerate(comparison_results):
    def fmt_prec(p):
        return f"{p:.0%}" if p is not None else "n/a"

    print(
        f"{i+1:<3} {r['question'][:38]:<40} {str(r['expected'] or 'any'):<10} "
        f"{fmt_prec(r['naive_precision']):>7} {fmt_prec(r['kw_precision']):>8} "
        f"{fmt_prec(r['smart_precision']):>7}  {r['smart_method']}"
    )

# Compute averages (excluding None values)
def safe_avg(values):
    valid = [v for v in values if v is not None]
    return sum(valid) / len(valid) if valid else 0.0

print(f"\n{'AVERAGE':<53} "
      f"{safe_avg([r['naive_precision'] for r in comparison_results]):>7.0%} "
      f"{safe_avg([r['kw_precision'] for r in comparison_results]):>8.0%} "
      f"{safe_avg([r['smart_precision'] for r in comparison_results]):>7.0%}")

## Section 5: Written Analysis (30 pts)

Answer each question below. Use specific examples from your comparison results.
Reference query numbers and actual precision values -- do not speak in generalities.

### Q3 (10 pts)

Pick the query where your LLM pipeline most clearly outperformed the other approaches.
Walk through: what did the LLM classify it as? What confidence? What routing decision?
What did the other approaches do differently? Why did yours win?

*Your answer here*

### Q4 (8 pts)

Pick a query where your pipeline made a bad decision -- or at least a questionable one.
What went wrong: the classification, the confidence assessment, or the routing logic?
How would you fix it?

*Your answer here*

### Q5 (7 pts)

Every LLM classification call costs tokens and adds latency. Was the added intelligence
worth the cost? How would you measure "worth it" in production? Is there a case where you
would fall back to keyword matching?

*Your answer here*

### Q6 (5 pts)

Some queries legitimately span multiple document types. For example: *"What budget items were
discussed at the board meeting?"* touches both financial and meeting documents. How would you
modify your classification schema and routing logic to handle this? Describe your approach --
you do not need to implement it.

*Your answer here*

## Section 6: Observability (After Session 4.1) -- Optional Bonus

After Session 4.1, come back and add PipelineLogger integration to your evaluation runner.
This is optional -- your lab grade is based on Sections 1-5 above.

*This section will be covered in Session 4.1. Leave it for now.*

In [ ]:
# After Session 4.1, uncomment and integrate PipelineLogger:
#
# from src.s5_observability.logger import PipelineLogger
#
# logger = PipelineLogger("lab2_evaluation")
#
# Add logging calls inside smart_retrieve():
#   logger.log_retrieval(question, sources, scores, latency)
#   logger.log_generation(question, answer, input_tokens, output_tokens)
#
# Then display the log summary:
#   logger.summary()

print("PipelineLogger integration -- complete after Session 4.1")

## Before You Submit

- [ ] `llm_classify()` working with structured output
- [ ] `smart_retrieve()` working with confidence-based routing
- [ ] System prompt updated with confidence definitions
- [ ] All 8 queries run through all 3 approaches
- [ ] Comparison results documented
- [ ] Q1-Q6 answered with specific examples and reasoning
- [ ] Reasoning explains architectural decisions, not just results